# Passo 5: Analisando o tráfego de rede (TCP/UDP) com TCPDump

Essa é uma das análises mais ricas que você pode fazer! O RTSP (Real Time Streaming Protocol) é um protocolo incrível porque ele divide a comunicação:
1. Ele usa o **TCP** (porta 554) para 'conversar' com a câmera: pedir play, pausar, enviar o usuário e senha.
2. Ele usa o **UDP** (portas aleatórias) para despejar os pacotes de vídeo pesados (RTP/H.264), porque o UDP não exige confirmação de recebimento (se perder um frame, segue a vida, não dá lag).

Eu já instalei o `tcpdump` rodando silenciosamente no seu Docker! Vamos rodá-lo junto com a captura.

In [1]:
import subprocess
import cv2
import time

IP_CAMERA = '10.64.0.61'
# Cole a sua URL aqui
RTSP_URL = f'rtsp://admin:@Jeff2712@{IP_CAMERA}:554/'

# --- 1. INICIANDO O TCPDUMP NO FUNDO ---
# Estamos dizendo: escute tudo que vai/vem da câmera e salve no arquivo .pcap
print('Iniciando o sniffer de rede (TCPDump)...')
processo_tcpdump = subprocess.Popen([
    'tcpdump',
    '-i', 'any',          # Qualquer interface de rede dentro do docker
    'host', IP_CAMERA,    # Focar apenas no IP da câmera
    '-w', 'analise_camera.pcap' # Salvar para podermos abrir no Wireshark
])

# Damos 2 segundinhos pro tcpdump começar a escutar bem
time.sleep(2)

Iniciando o sniffer de rede (TCPDump)...


tcpdump: WARNING: any: That device doesn't support promiscuous mode
(Promiscuous mode not supported on the "any" device)
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes


In [2]:
# --- 2. GERANDO O TRÁFEGO (Puxando vídeo) ---
print('Conectando na câmera para gerar tráfego de vídeo...')
cap = cv2.VideoCapture(RTSP_URL)

if cap.isOpened():
    print('Conectado! Puxando 150 pacotes (frames)...')
    for i in range(150):
        ret, frame = cap.read()
        if not ret:
            break
else:
    print('Não conectou.')

cap.release()
print('Fim da captura do OpenCV.')

Conectando na câmera para gerar tráfego de vídeo...
Conectado! Puxando 150 pacotes (frames)...
Fim da captura do OpenCV.


In [3]:
# --- 3. FINALIZANDO E SALVANDO O ARQUIVO DE REDE ---
print('Encerrando o TCPDump e salvando o arquivo .pcap...')
processo_tcpdump.terminate() 
processo_tcpdump.wait()
print('Salvo! Verifique a barra lateral pelos arquivos.')

Encerrando o TCPDump e salvando o arquivo .pcap...
Salvo! Verifique a barra lateral pelos arquivos.


746 packets captured
864 packets received by filter
0 packets dropped by kernel


### O que fazer com o arquivo `analise_camera.pcap`?

1. Vá na barra lateral esquerda e faça o **Download** do arquivo `analise_camera.pcap` para o seu computador.
2. Abra ele usando o software gratuito **Wireshark** (se você tiver instalado aí no seu PC físico).
3. No Wireshark, digite `rtsp` na barra verde de filtro lá em cima.
   * Você verá os pacotes `DESCRIBE`, `SETUP` e `PLAY` (é o seu código falando com a câmera).
   * Se você limpar o filtro, verá uma enxurrada absurda de pacotes `UDP` marcados como protocolo `RTP`. Esses são os pequenos blocos de H.264 da imagem real da sua bancada fluindo pela sua rede!